# FreeNeRF on Kaggle (launcher only: all logic lives in the repository)
**Before running:** right panel -> Accelerator: GPU (T4/P100), Internet: On. Add-ons -> Secrets: `GH_TOKEN`
(fine-grained GitHub token, this repo only, Contents: read-only). After the last cell: **Save Version -> Save & Run All (Commit)**
so the run continues with the tab closed. Re-running is safe: finished runs are skipped.

In [ ]:
from kaggle_secrets import UserSecretsClient
import os
os.environ["GH_TOKEN"] = UserSecretsClient().get_secret("GH_TOKEN")

In [ ]:
!git clone -q https://$GH_TOKEN@github.com/warismittal/freenerf-nerfstudio.git
%cd freenerf-nerfstudio
!pip install -q nerfstudio lpips optuna pytest
!pip install -q -e .
!pip freeze > requirements.cuda.lock
import torch; print(torch.__version__, "cuda:", torch.cuda.is_available())   # must be True; if nerfstudio replaced torch, reinstall a CUDA build

## Data (NeRF-synthetic -> 400x400 -> hashed manifests)
If the automatic download fails, attach the dataset as a Kaggle input and run `!ln -s /kaggle/input/<dataset>/nerf_synthetic data/blender` first.

In [ ]:
!bash scripts/get_data.sh

## Gate 1: smoke test, then time one pilot run (tau)

In [ ]:
!bash scripts/smoke_test.sh

In [ ]:
%%time
# tau: wall-clock of ONE 8-view vanilla run at the iteration count you intend to use. Write it into README.md and the proposal's budget rule.
!python scripts/run_arm.py --scene lego --arm A --n 8 --seed 0 --iters 30000 --device cuda

In [ ]:
!python scripts/run_arm.py --scene lego --arm A --n dense --seed 0 --iters 30000 --device cuda   # control C4 and the denominator of rho

In [ ]:
!python scripts/run_controls.py

## Experiments (set I from the budget rule: 82 x tau(I) <= 0.8 x your GPU hours)

In [ ]:
!python scripts/run_all.py --list

In [ ]:
!bash scripts/run_all.sh --only E1 --iters 30000 --device cuda

## Save everything that matters (Kaggle keeps /kaggle/working as the versioned output)

In [ ]:
!python -m evalkit.aggregate && zip -qr /kaggle/working/results.zip results manifests configs && ls -la /kaggle/working/results.zip